# 06 Shifted GroundingDINO/SAM Mask Probe

역할: condition shift가 걸린 정상 이미지에서 GroundingDINO+SAM이 component mask를 계속 잘 찾는지 소수 샘플로 확인한다. 기존 graph probe처럼 precomputed clean mask를 그대로 쓰지 않고, shifted image를 UniVAD grounding segmentation에 다시 넣어 새 mask를 생성한 뒤 clean/reference mask와 비교한다.


## Cell Role: Runtime Guard

GroundingDINO/SAM 경로는 torch와 Transformers를 import한다. Colab 런타임에서 NumPy/Pandas ABI가 torch 및 notebook 의존성과 충돌할 수 있으므로, NumPy 1.26.4와 호환 Pandas로 맞춘 뒤 런타임을 재시작한다.


In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

TARGET_NUMPY = '1.26.4'
TARGET_PANDAS = '2.2.3'


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return '-'


def pandas_runtime_ready() -> tuple[bool, str]:
    try:
        import pandas as pd  # noqa: F401
        return True, package_version('pandas')
    except Exception as exc:  # pandas/numpy ABI mismatch also lands here
        return False, f'{type(exc).__name__}: {exc}'


numpy_version = package_version('numpy')
pandas_ready, pandas_status = pandas_runtime_ready()

print('numpy =', numpy_version)
print('pandas =', pandas_status)

needs_repair = (
    numpy_version == '-'
    or numpy_version.split('.', maxsplit=1)[0] != '1'
    or not pandas_ready
)
if needs_repair:
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade',
            '--force-reinstall',
            '--no-cache-dir',
            f'numpy=={TARGET_NUMPY}',
            f'pandas=={TARGET_PANDAS}',
        ],
        check=True,
    )
    raise SystemExit(
        'Installed compatible numpy/pandas runtime for shifted grounding probe. '
        'Restart the Colab runtime once, then rerun this notebook from the top.'
    )
print('numpy/pandas runtime is compatible with this notebook stack')


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM` checkout을 사용하고, 로컬에서는 현재 repo root를 찾는다. GroundingDINO tokenizer 경로에서 TensorFlow backend가 import되지 않도록 Transformers guard를 먼저 적용한다.


In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'


def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )

colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(
            [
                'git',
                '-C',
                str(colab_checkout),
                'checkout',
                'FETCH_HEAD',
                '--',
                'experiments/validation/condition_shift_baseline/src',
                'experiments/validation/condition_shift_baseline/configs',
                'experiments/validation/condition_shift_baseline/notebook/06_shifted_grounding_probe.ipynb',
            ],
            check=True,
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((candidate.resolve() for candidate in repo_candidates if candidate.exists() and is_regram_root(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
ORCHESTRATION_ROOT = SRC_ROOT / 'orchestration'
for import_root in [SRC_ROOT, ORCHESTRATION_ROOT]:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from univad.transformers_runtime import disable_transformers_tensorflow_backend

disable_transformers_tensorflow_backend()

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Probe Settings And Readiness

샘플 수를 작게 유지한다. raw LOCO와 clean grounding masks는 Drive cache에서 복구하고, SAM/DINO checkpoint는 shared model checkpoint cache에서 복구한다. 외부 UniVAD repo나 editable GroundingDINO가 없으면 01 setup notebook을 먼저 실행해야 한다.


In [ ]:
CATEGORY = 'breakfast_box'
SAMPLE_IDS = ['000.png', '001.png', '002.png']
SEGMENTATION_BACKEND = 'robustsam'  # 'sam_hq' or 'robustsam'
AUTO_CLONE_UNIVAD_REPO = True
UNIVAD_REPO_URL = 'https://github.com/FantasticGNU/UniVAD.git'
AUTO_CLONE_ROBUSTSAM_REPO = True
ROBUSTSAM_REPO_URL = 'https://github.com/robustsam/RobustSAM.git'
ROBUSTSAM_MODEL_SIZE = 'b'  # 'b', 'l', or 'h'; b is fastest for probe
ROBUSTSAM_CHECKPOINT_URL = 'https://huggingface.co/robustsam/robustsam/resolve/main/model_checkpoint/robustsam_checkpoint_b.pth'
SHIFT_SPECS = [
    ('brightness', 'high'),
    ('gaussian_blur', 'high'),
    ('gaussian_noise', 'high'),
    ('position_shift', 'high'),
]
FORCE_REGENERATE = False

RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
GROUNDING_MASK_DATA_ROOT = RAW_LOCO_ROOT
GROUNDING_MASK_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'masks' / 'mvtec_loco_caption'
UNIVAD_ROOT = REPO_ROOT / 'external' / 'UniVAD'
UNIVAD_CHECKPOINT_ROOT = UNIVAD_ROOT / 'pretrained_ckpts'
SAM_CHECKPOINT = UNIVAD_CHECKPOINT_ROOT / 'sam_hq_vit_h.pth'
DINO_CHECKPOINT = UNIVAD_CHECKPOINT_ROOT / 'groundingdino_swint_ogc.pth'
ROBUSTSAM_ROOT = REPO_ROOT / 'external' / 'RobustSAM'
ROBUSTSAM_CHECKPOINT_ROOT = REPO_ROOT / 'external' / 'model_checkpoints' / 'robustsam'
ROBUSTSAM_CHECKPOINT = ROBUSTSAM_CHECKPOINT_ROOT / f'robustsam_checkpoint_{ROBUSTSAM_MODEL_SIZE}.pth'

PROBE_ROOT = EXP_ROOT / 'reports' / 'shifted_grounding_probe'
SHIFTED_IMAGE_ROOT = PROBE_ROOT / 'images'
GENERATED_MASK_ROOT = PROBE_ROOT / 'generated_masks' / SEGMENTATION_BACKEND
METRIC_CSV = PROBE_ROOT / f'{CATEGORY}_{SEGMENTATION_BACKEND}_dino_shift_mask_metrics.csv'
GROUNDING_CLUSTER_CONFIG = EXP_ROOT / 'configs' / 'grounding_mask_cluster.yaml'
COMPONENT_SUMMARY_ROOT = PROBE_ROOT / 'component_summaries'
CLEAN_COMPONENT_SUMMARY_PATH = COMPONENT_SUMMARY_ROOT / f'{CATEGORY}_clean_component_summary.json'
SHIFT_COMPONENT_SCORE_CSV = PROBE_ROOT / f'{CATEGORY}_{SEGMENTATION_BACKEND}_generated_shift_component_scores.csv'
SHIFT_COMPONENT_SCORE_SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_{SEGMENTATION_BACKEND}_generated_shift_component_score_summary.csv'
COMPONENT_QUALITY_CSV = PROBE_ROOT / f'{CATEGORY}_{SEGMENTATION_BACKEND}_generated_shift_component_quality.csv'
PROBE_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')
DRIVE_MASK_ROOT = Path('/content/drive/MyDrive/ReGraM/univad_masks/mvtec_loco_caption')
DRIVE_MODEL_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/ReGraM/model_checkpoints')


def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')


def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [
        parent / 'content' / dataset_name,
        parent / 'data' / 'row' / dataset_name,
    ]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct


def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT, GROUNDING_MASK_DATA_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    GROUNDING_MASK_DATA_ROOT = RAW_LOCO_ROOT
    if (RAW_LOCO_ROOT / CATEGORY / 'test' / 'good').exists():
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    GROUNDING_MASK_DATA_ROOT = RAW_LOCO_ROOT


def expected_clean_mask_path(source_path: Path, *, mask_root: Path) -> Path:
    try:
        rel_path = source_path.relative_to(GROUNDING_MASK_DATA_ROOT / CATEGORY)
    except ValueError:
        parts = list(source_path.parts)
        category_index = len(parts) - 1 - parts[::-1].index(CATEGORY)
        rel_path = Path(*parts[category_index + 1 :])
    return mask_root / CATEGORY / rel_path.with_suffix('') / 'grounding_mask.png'


def restore_clean_masks_from_drive_if_needed(source_paths: list[Path]) -> None:
    missing = [path for path in source_paths if not expected_clean_mask_path(path, mask_root=GROUNDING_MASK_ROOT).exists()]
    if not missing:
        return
    mount_drive_if_available()
    copied = 0
    for source_path in missing:
        session_mask = expected_clean_mask_path(source_path, mask_root=GROUNDING_MASK_ROOT)
        drive_mask = expected_clean_mask_path(source_path, mask_root=DRIVE_MASK_ROOT)
        if not drive_mask.exists():
            continue
        session_mask.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_mask, session_mask)
        copied += 1
    print(f'restored clean masks from Drive: {copied}/{len(missing)}')


def restore_checkpoints_from_drive_if_needed() -> None:
    mount_drive_if_available()
    for filename in ('sam_hq_vit_h.pth', 'groundingdino_swint_ogc.pth'):
        session_path = UNIVAD_CHECKPOINT_ROOT / filename
        drive_path = DRIVE_MODEL_CHECKPOINT_ROOT / filename
        if session_path.exists() or not drive_path.exists():
            continue
        session_path.parent.mkdir(parents=True, exist_ok=True)
        print(f'restore checkpoint from Drive: {drive_path} -> {session_path}')
        shutil.copy2(drive_path, session_path)

    robust_drive_candidates = [
        DRIVE_MODEL_CHECKPOINT_ROOT / 'robustsam' / ROBUSTSAM_CHECKPOINT.name,
        DRIVE_MODEL_CHECKPOINT_ROOT / ROBUSTSAM_CHECKPOINT.name,
    ]
    if not ROBUSTSAM_CHECKPOINT.exists():
        for drive_path in robust_drive_candidates:
            if not drive_path.exists():
                continue
            ROBUSTSAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
            print(f'restore RobustSAM checkpoint from Drive: {drive_path} -> {ROBUSTSAM_CHECKPOINT}')
            shutil.copy2(drive_path, ROBUSTSAM_CHECKPOINT)
            break


def download_robustsam_checkpoint_if_needed() -> None:
    if SEGMENTATION_BACKEND != 'robustsam' or ROBUSTSAM_CHECKPOINT.exists():
        return
    ROBUSTSAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    print(f'download RobustSAM checkpoint: {ROBUSTSAM_CHECKPOINT_URL} -> {ROBUSTSAM_CHECKPOINT}')
    subprocess.run(['curl', '-L', '--fail', '-o', str(ROBUSTSAM_CHECKPOINT), ROBUSTSAM_CHECKPOINT_URL], check=True)


def ensure_robustsam_repo_if_needed() -> None:
    if SEGMENTATION_BACKEND != 'robustsam' or (ROBUSTSAM_ROOT / 'robust_segment_anything').exists():
        return
    if not AUTO_CLONE_ROBUSTSAM_REPO:
        return
    ROBUSTSAM_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(f'clone RobustSAM: {ROBUSTSAM_REPO_URL} -> {ROBUSTSAM_ROOT}')
    subprocess.run(['git', 'clone', ROBUSTSAM_REPO_URL, str(ROBUSTSAM_ROOT)], check=True)


def _copy_tree_merge(src_dir: Path, dst_dir: Path) -> None:
    dst_dir.mkdir(parents=True, exist_ok=True)
    for item in src_dir.iterdir():
        destination = dst_dir / item.name
        if item.is_dir():
            _copy_tree_merge(item, destination)
        elif not destination.exists():
            shutil.copy2(item, destination)


def ensure_univad_repo_if_needed() -> None:
    key_paths = [
        UNIVAD_ROOT / 'models' / 'GroundingDINO',
        UNIVAD_ROOT / 'models' / 'segment_anything',
        UNIVAD_ROOT / 'configs' / 'class_histogram' / f'{CATEGORY}.yaml',
    ]
    if all(path.exists() for path in key_paths):
        return
    if not AUTO_CLONE_UNIVAD_REPO:
        return
    if (UNIVAD_ROOT / '.git').exists():
        print(f'update UniVAD submodules: {UNIVAD_ROOT}')
        subprocess.run(['git', '-C', str(UNIVAD_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True)
        return
    if not UNIVAD_ROOT.exists() or not any(UNIVAD_ROOT.iterdir()):
        UNIVAD_ROOT.parent.mkdir(parents=True, exist_ok=True)
        print(f'clone UniVAD recursively: {UNIVAD_REPO_URL} -> {UNIVAD_ROOT}')
        subprocess.run(['git', 'clone', '--recurse-submodules', UNIVAD_REPO_URL, str(UNIVAD_ROOT)], check=True)
        return

    temp_clone = UNIVAD_ROOT.parent / '_tmp_univad_clone'
    if temp_clone.exists():
        shutil.rmtree(temp_clone)
    print(f'restore UniVAD repo files into partial directory: {UNIVAD_ROOT}')
    subprocess.run(['git', 'clone', '--recurse-submodules', UNIVAD_REPO_URL, str(temp_clone)], check=True)
    try:
        for rel_name in ['configs', 'models']:
            _copy_tree_merge(temp_clone / rel_name, UNIVAD_ROOT / rel_name)
        for file_name in ['UniVAD.py', 'requirements.txt', 'README.md']:
            source_file = temp_clone / file_name
            destination_file = UNIVAD_ROOT / file_name
            if source_file.exists() and not destination_file.exists():
                destination_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source_file, destination_file)
    finally:
        if temp_clone.exists():
            shutil.rmtree(temp_clone)

restore_raw_loco_from_drive_if_needed()
restore_checkpoints_from_drive_if_needed()
ensure_univad_repo_if_needed()
ensure_robustsam_repo_if_needed()
download_robustsam_checkpoint_if_needed()

required_paths = {
    'raw LOCO category': RAW_LOCO_ROOT / CATEGORY / 'test' / 'good',
    'UniVAD repo': UNIVAD_ROOT,
    'GroundingDINO package': UNIVAD_ROOT / 'models' / 'GroundingDINO',
    'GroundingDINO checkpoint': DINO_CHECKPOINT,
    'class histogram config': UNIVAD_ROOT / 'configs' / 'class_histogram' / f'{CATEGORY}.yaml',
}
if SEGMENTATION_BACKEND == 'sam_hq':
    required_paths.update({
        'segment_anything package': UNIVAD_ROOT / 'models' / 'segment_anything',
        'SAM-HQ checkpoint': SAM_CHECKPOINT,
    })
elif SEGMENTATION_BACKEND == 'robustsam':
    required_paths.update({
        'RobustSAM package': ROBUSTSAM_ROOT / 'robust_segment_anything',
        'RobustSAM checkpoint': ROBUSTSAM_CHECKPOINT,
    })
else:
    raise ValueError(f'Unsupported SEGMENTATION_BACKEND={SEGMENTATION_BACKEND}')
missing_paths = {name: path for name, path in required_paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(f'Missing shifted grounding probe paths. Run 01 setup first: {missing_paths}')

settings = {
    'category': CATEGORY,
    'sample_ids': ', '.join(SAMPLE_IDS),
    'shift_specs': ', '.join(f'{aug}/{sev}' for aug, sev in SHIFT_SPECS),
    'raw_loco_root': str(RAW_LOCO_ROOT),
    'clean_mask_root': str(GROUNDING_MASK_ROOT),
    'generated_mask_root': str(GENERATED_MASK_ROOT),
    'segmentation_backend': SEGMENTATION_BACKEND,
    'robustsam_model_size': ROBUSTSAM_MODEL_SIZE if SEGMENTATION_BACKEND == 'robustsam' else '-',
    'robustsam_root': str(ROBUSTSAM_ROOT) if SEGMENTATION_BACKEND == 'robustsam' else '-',
    'robustsam_checkpoint': str(ROBUSTSAM_CHECKPOINT) if SEGMENTATION_BACKEND == 'robustsam' else '-',
    'metric_csv': str(METRIC_CSV),
    'auto_clone_univad_repo': AUTO_CLONE_UNIVAD_REPO,
    'univad_repo_url': UNIVAD_REPO_URL,
}
display(pd.DataFrame([settings]))


## Cell Role: Build Shifted Sample Images

manifest에서 sample row를 고르고, 원본 정상 이미지를 지정된 shift 조건으로 변환해 probe 전용 이미지 폴더에 저장한다. `base_image_id`를 보존해서 clean/shift 결과를 같은 이미지 기준으로 묶는다.


In [ ]:
from data.augmentation_runtime import apply_augmentation, load_manifest
from PIL import Image


def resolve_source_path(entry: dict) -> Path:
    path = Path(entry['source_path'])
    if entry.get('source_path_mode') == 'repo_relative' or not path.is_absolute():
        return REPO_ROOT / path
    return path


def select_manifest_entries(augmentation_type: str, severity: str, sample_ids: list[str]) -> list[dict]:
    manifest_path = REPO_ROOT / 'manifests' / f'query_{augmentation_type}.jsonl'
    entries = [
        entry
        for entry in load_manifest(manifest_path)
        if entry.get('category') == CATEGORY
        and entry.get('augmentation_type') == augmentation_type
        and entry.get('severity') == severity
    ]
    by_source = {entry['source_id']: entry for entry in entries}
    selected = [by_source[source_id] for source_id in sample_ids if source_id in by_source]
    if selected:
        return selected
    return entries[:len(sample_ids)]

probe_rows = []
source_paths_by_id: dict[str, Path] = {}
for augmentation_type, severity in SHIFT_SPECS:
    condition = f'{augmentation_type}_{severity}'
    for entry in select_manifest_entries(augmentation_type, severity, SAMPLE_IDS):
        source_path = resolve_source_path(entry)
        source_paths_by_id[entry['source_id']] = source_path
        image = Image.open(source_path).convert('RGB')
        shifted = apply_augmentation(
            image,
            augmentation_type=augmentation_type,
            severity=severity,
            seed=int(entry['seed']),
            params=entry.get('params'),
        )
        shifted_path = SHIFTED_IMAGE_ROOT / CATEGORY / condition / entry['source_id']
        shifted_path.parent.mkdir(parents=True, exist_ok=True)
        shifted.save(shifted_path)
        probe_rows.append(
            {
                'base_image_id': entry['source_id'],
                'condition': condition,
                'augmentation_type': augmentation_type,
                'severity': severity,
                'seed': int(entry['seed']),
                'params': entry.get('params') or {},
                'source_path': str(source_path),
                'shifted_path': str(shifted_path),
            }
        )

if not probe_rows:
    raise ValueError('No probe rows selected')

restore_clean_masks_from_drive_if_needed(list(source_paths_by_id.values()))
probe_df = pd.DataFrame(probe_rows)
display(probe_df)


## Cell Role: Generate GroundingDINO/SAM Masks On Shifted Images

shifted image를 UniVAD `grounding_segmentation`에 넣어 새 `grounding_mask.png`를 생성한다. 이미 생성된 mask가 있으면 `FORCE_REGENERATE=False`일 때 재사용한다.


In [ ]:
import os
import time

import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for GroundingDINO/SAM mask generation. Use a GPU Colab runtime.')

for import_root in [SRC_ROOT, ORCHESTRATION_ROOT, UNIVAD_ROOT, UNIVAD_ROOT / 'models' / 'GroundingDINO']:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from univad.grounding_backends import grounding_segmentation_with_backend
from univad.setup_runtime import (
    default_univad_setup_flags,
    ensure_editable_package,
    maybe_fix_univad_runtime_dependency_stack,
    maybe_fix_univad_transformers_stack,
)

runtime_settings = default_univad_setup_flags()
runtime_dependency_status = maybe_fix_univad_runtime_dependency_stack(runtime_settings)
transformers_status = maybe_fix_univad_transformers_stack(runtime_settings)
for dependency_name, dependency_status in {
    'runtime_dependencies': runtime_dependency_status,
    'transformers': transformers_status,
}.items():
    if dependency_status.get('error') not in (None, '-', ''):
        raise RuntimeError(f'UniVAD {dependency_name} setup failed: {dependency_status}')
    if dependency_status.get('restart_required'):
        raise RuntimeError(
            'Installed or updated UniVAD dependencies for shifted grounding probe. '
            'Restart the runtime once, then rerun this notebook from the top. '
            f'Detail: {dependency_name} -> {dependency_status.get("note", "restart required")}'
        )


def unload_groundingdino_modules() -> None:
    for module_name in list(sys.modules):
        if (
            module_name == 'groundingdino'
            or module_name.startswith('groundingdino.')
            or module_name in {'models.component_segmentaion', 'models.grounded_sam'}
        ):
            sys.modules.pop(module_name, None)


def groundingdino_cuda_extension_ready() -> tuple[bool, str]:
    try:
        import groundingdino._C  # noqa: F401
        return True, 'ready'
    except Exception as exc:
        return False, f'{type(exc).__name__}: {exc}'


groundingdino_dir = UNIVAD_ROOT / 'models' / 'GroundingDINO'
extension_ready, extension_status = groundingdino_cuda_extension_ready()
if not extension_ready:
    print(f'install editable GroundingDINO package: {groundingdino_dir} ({extension_status})')
    ensure_editable_package(groundingdino_dir)
    unload_groundingdino_modules()
    extension_ready, extension_status = groundingdino_cuda_extension_ready()
if not extension_ready:
    raise RuntimeError(
        'GroundingDINO CUDA extension is not importable after editable install. '
        'Restart the runtime once and rerun this notebook from the top. '
        f'Detail: {extension_status}'
    )

os.environ.setdefault('TORCH_HOME', str(UNIVAD_CHECKPOINT_ROOT / 'torch'))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

with (UNIVAD_ROOT / 'configs' / 'class_histogram' / f'{CATEGORY}.yaml').open('r', encoding='utf-8') as handle:
    grounding_config = yaml.safe_load(handle)['grounding_config']


def find_generated_mask_path(image_path: Path, output_dir: Path) -> Path | None:
    candidates = sorted(output_dir.rglob(f'{image_path.stem}/grounding_mask.png'))
    if not candidates:
        candidates = sorted(output_dir.rglob('grounding_mask.png'))
        candidates = [candidate for candidate in candidates if candidate.parent.name == image_path.stem]
    if not candidates:
        return None
    return candidates[0]

mask_rows = []
previous_cwd = Path.cwd()
try:
    os.chdir(UNIVAD_ROOT)
    for condition, group in probe_df.groupby('condition'):
        output_dir = GENERATED_MASK_ROOT / CATEGORY / condition
        shifted_paths = [Path(path) for path in group['shifted_path'].tolist()]
        expected = [find_generated_mask_path(path, output_dir) for path in shifted_paths]
        needs_generation = FORCE_REGENERATE or any(path is None or not path.exists() for path in expected)
        if needs_generation:
            started_at = time.perf_counter()
            print(f'generate shifted grounding masks: backend={SEGMENTATION_BACKEND} condition={condition} images={len(shifted_paths)} output={output_dir}')
            grounding_segmentation_with_backend(
                [str(path) for path in shifted_paths],
                str(output_dir),
                grounding_config,
                backend=SEGMENTATION_BACKEND,
                robustsam_root=ROBUSTSAM_ROOT,
                robustsam_checkpoint=ROBUSTSAM_CHECKPOINT,
                robustsam_model_size=ROBUSTSAM_MODEL_SIZE,
            )
            print(f'done condition={condition} sec={time.perf_counter() - started_at:.1f}')
        for _, row in group.iterrows():
            shifted_path = Path(row['shifted_path'])
            generated_mask_path = find_generated_mask_path(shifted_path, output_dir)
            mask_rows.append({**row.to_dict(), 'generated_mask_path': str(generated_mask_path) if generated_mask_path else '-'})
finally:
    os.chdir(previous_cwd)

mask_df = pd.DataFrame(mask_rows)
display(mask_df[['base_image_id', 'condition', 'shifted_path', 'generated_mask_path']])
missing_generated = mask_df[mask_df['generated_mask_path'].eq('-')]
if not missing_generated.empty:
    raise FileNotFoundError(f'Generated masks not found for rows: {missing_generated.to_dict(orient="records")}')


## Cell Role: Score Shifted Mask Quality

appearance shift는 clean mask를 그대로 reference로 쓰고, position shift는 clean mask를 같은 geometric transform으로 이동/축소한 expected mask와 비교한다. 주요 지표는 foreground IoU, coverage ratio, generated label count 변화다.


In [ ]:
import numpy as np

POSITION_CORNER_PLACEMENTS = ('top_left', 'top_right', 'bottom_left', 'bottom_right')


def load_rgb(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert('RGB'))


def foreground_mask(label_rgb: np.ndarray) -> np.ndarray:
    return np.any(label_rgb != 0, axis=-1)


def label_count(label_rgb: np.ndarray) -> int:
    colors = np.unique(label_rgb.reshape(-1, 3), axis=0)
    return int(sum(1 for color in colors if not np.all(color == 0)))


def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(intersection / union) if union else 1.0


def position_shift_expected_mask(clean_mask_image: Image.Image, params: dict, seed: int) -> Image.Image:
    width, height = clean_mask_image.size
    center_shift_ratio = float(params.get('center_shift_ratio', params.get('max_ratio', 0.0)))
    scale = float(params.get('scale', 1.0 - (2.0 * center_shift_ratio)))
    scale = min(1.0, max(0.05, scale))
    shifted_size = (
        max(1, min(width, int(round(width * scale)))),
        max(1, min(height, int(round(height * scale)))),
    )
    placement = params.get('placement', params.get('anchor', 'seeded_corner'))
    if placement in {'seeded_corner', 'random_corner'}:
        placement = random.Random(seed).choice(POSITION_CORNER_PLACEMENTS)
    max_x = width - shifted_size[0]
    max_y = height - shifted_size[1]
    offsets = {
        'top_left': (0, 0),
        'top_right': (max_x, 0),
        'bottom_left': (0, max_y),
        'bottom_right': (max_x, max_y),
        'center': (max_x // 2, max_y // 2),
    }
    offset = offsets[str(placement)]
    shifted = clean_mask_image.convert('RGB').resize(shifted_size, resample=Image.Resampling.NEAREST)
    canvas = Image.new('RGB', clean_mask_image.size, (0, 0, 0))
    canvas.paste(shifted, offset)
    return canvas

metric_rows = []
for _, row in mask_df.iterrows():
    source_path = Path(row['source_path'])
    clean_mask_path = expected_clean_mask_path(source_path, mask_root=GROUNDING_MASK_ROOT)
    clean_mask_image = Image.open(clean_mask_path).convert('RGB')
    generated_mask_image = Image.open(Path(row['generated_mask_path'])).convert('RGB')
    if row['augmentation_type'] == 'position_shift':
        expected_mask_image = position_shift_expected_mask(clean_mask_image, row['params'], int(row['seed']))
    else:
        expected_mask_image = clean_mask_image

    expected_rgb = np.asarray(expected_mask_image.convert('RGB'))
    generated_rgb = np.asarray(generated_mask_image.convert('RGB'))
    expected_fg = foreground_mask(expected_rgb)
    generated_fg = foreground_mask(generated_rgb)
    expected_area = int(expected_fg.sum())
    generated_area = int(generated_fg.sum())
    metric_rows.append(
        {
            'base_image_id': row['base_image_id'],
            'condition': row['condition'],
            'augmentation_type': row['augmentation_type'],
            'severity': row['severity'],
            'clean_mask_path': str(clean_mask_path),
            'generated_mask_path': row['generated_mask_path'],
            'expected_area': expected_area,
            'generated_area': generated_area,
            'foreground_iou': binary_iou(expected_fg, generated_fg),
            'coverage_ratio': float(generated_area / expected_area) if expected_area else 0.0,
            'expected_label_count': label_count(expected_rgb),
            'generated_label_count': label_count(generated_rgb),
            'label_count_delta': label_count(generated_rgb) - label_count(expected_rgb),
        }
    )

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(METRIC_CSV, index=False)
print(f'saved metrics: {METRIC_CSV}')
display(metrics_df)
display(metrics_df.groupby('condition')[['foreground_iou', 'coverage_ratio', 'label_count_delta']].agg(['mean', 'min', 'max']))


## Cell Role: Cluster Generated Masks Into Component Nodes

clean mask와 shifted image에서 새로 생성한 GroundingDINO/SAM mask를 같은 `grounding_mask_cluster` 규칙으로 thing / stuff_cluster / small_isolated component node로 정규화한다. 이 단계는 segmentation 성능을 주장하려는 것이 아니라, shift가 걸려도 graph input component 구성이 유지되는지 보는 probe다. position shift는 실제 geometry가 변하므로 graph score가 커질 수 있다.


In [ ]:
from graph_probe.batch_compare import compare_summary_conditions, summarize_condition_scores
from graph_probe.component_io import component_quality_row, save_json
from graph_probe.component_summary import load_config_file
from relation.grounding_mask_cluster import cluster_masks_to_components, raw_masks_from_label_image

cluster_config = load_config_file(GROUNDING_CLUSTER_CONFIG)
COMPONENT_SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)


def node_type_counts(nodes: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for node in nodes:
        node_type = str(node.get('node_type', '-'))
        counts[node_type] = counts.get(node_type, 0) + 1
    return counts


def build_component_row(
    *,
    image_path: Path,
    mask_path: Path,
    base_image_id: str,
    condition: str,
    augmentation_type: str,
    severity: str,
    component_source: str,
) -> tuple[dict, dict]:
    image = Image.open(image_path).convert('RGB')
    with Image.open(mask_path) as mask_image:
        raw_masks = raw_masks_from_label_image(mask_image)
    nodes = cluster_masks_to_components(
        np.asarray(image),
        raw_masks,
        config=cluster_config,
        category=CATEGORY,
    )
    counts = node_type_counts(nodes)
    summary_row = {
        'base_image_id': base_image_id,
        'source_id': base_image_id,
        'image_id': base_image_id,
        'source_path': str(image_path),
        'mask_path': str(mask_path),
        'category': CATEGORY,
        'condition': condition,
        'augmentation_type': augmentation_type,
        'severity': severity,
        'component_model': 'grounding_mask_cluster',
        'component_source': component_source,
        'component_count': len(nodes),
        'raw_mask_count': len(raw_masks),
        'thing_count': counts.get('thing', 0),
        'stuff_cluster_count': counts.get('stuff_cluster', 0),
        'small_isolated_count': counts.get('small_isolated', 0),
        'image_width': image.size[0],
        'image_height': image.size[1],
        'component_nodes': nodes,
    }
    quality_row = component_quality_row(
        nodes=nodes,
        image_id=base_image_id,
        category=CATEGORY,
        split='test/good',
        shift_type=condition,
        severity=severity,
        num_raw_masks=len(raw_masks),
    )
    quality_row.update(
        {
            'condition': condition,
            'component_source': component_source,
            'mask_path': str(mask_path),
        }
    )
    return summary_row, quality_row

clean_rows_by_id: dict[str, dict] = {}
quality_rows: list[dict] = []
for base_image_id, group in mask_df.groupby('base_image_id'):
    first = group.iloc[0]
    source_path = Path(first['source_path'])
    clean_mask_path = expected_clean_mask_path(source_path, mask_root=GROUNDING_MASK_ROOT)
    clean_row, clean_quality = build_component_row(
        image_path=source_path,
        mask_path=clean_mask_path,
        base_image_id=str(base_image_id),
        condition='clean_reference',
        augmentation_type='clean',
        severity='clean',
        component_source='clean_grounding_mask',
    )
    clean_rows_by_id[str(base_image_id)] = clean_row
    quality_rows.append(clean_quality)

clean_summary = {
    'probe': 'shifted_grounding_component_summary',
    'category': CATEGORY,
    'condition': 'clean_reference',
    'component_model': 'grounding_mask_cluster',
    'rows': list(clean_rows_by_id.values()),
}
save_json(clean_summary, CLEAN_COMPONENT_SUMMARY_PATH)

query_summary_paths: dict[str, Path] = {}
for condition, group in mask_df.groupby('condition'):
    shifted_rows = []
    for _, row in group.iterrows():
        shifted_row, shifted_quality = build_component_row(
            image_path=Path(row['shifted_path']),
            mask_path=Path(row['generated_mask_path']),
            base_image_id=str(row['base_image_id']),
            condition=str(condition),
            augmentation_type=str(row['augmentation_type']),
            severity=str(row['severity']),
            component_source='generated_shifted_grounding_mask',
        )
        shifted_rows.append(shifted_row)
        quality_rows.append(shifted_quality)
    summary_path = COMPONENT_SUMMARY_ROOT / f'{CATEGORY}_{condition}_generated_component_summary.json'
    save_json(
        {
            'probe': 'shifted_grounding_component_summary',
            'category': CATEGORY,
            'condition': condition,
            'component_model': 'grounding_mask_cluster',
            'rows': shifted_rows,
        },
        summary_path,
    )
    query_summary_paths[str(condition)] = summary_path

component_quality_df = pd.DataFrame(quality_rows)
component_quality_df.to_csv(COMPONENT_QUALITY_CSV, index=False)
component_score_df = compare_summary_conditions(
    CLEAN_COMPONENT_SUMMARY_PATH,
    query_summary_paths,
    category=CATEGORY,
    output_csv=SHIFT_COMPONENT_SCORE_CSV,
)
component_score_summary_df = summarize_condition_scores(
    component_score_df,
    output_csv=SHIFT_COMPONENT_SCORE_SUMMARY_CSV,
)

print(f'saved clean component summary: {CLEAN_COMPONENT_SUMMARY_PATH}')
print(f'saved component quality: {COMPONENT_QUALITY_CSV}')
print(f'saved component graph scores: {SHIFT_COMPONENT_SCORE_CSV}')
display(component_quality_df)
display(component_score_df)
display(component_score_summary_df)


## Cell Role: Visualize Component Stability

generated shifted mask에서 만든 component node 구성이 clean reference 대비 얼마나 바뀌는지 type 구성과 graph score decomposition으로 확인한다. bbox overlay는 clean/generated mask 위에 thing/stuff/small node가 어디 잡혔는지 빠르게 검토하기 위한 디버그 뷰다.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if 'LABEL_PALETTE' not in globals():
    LABEL_PALETTE = np.asarray([
        [31, 119, 180],
        [255, 127, 14],
        [44, 160, 44],
        [214, 39, 40],
        [148, 103, 189],
        [140, 86, 75],
        [227, 119, 194],
        [127, 127, 127],
        [188, 189, 34],
        [23, 190, 207],
    ], dtype=np.uint8)

if 'recolor_label_mask' not in globals():
    def recolor_label_mask(mask_rgb: np.ndarray) -> np.ndarray:
        flat = mask_rgb.reshape(-1, 3)
        colors, inverse = np.unique(flat, axis=0, return_inverse=True)
        recolored_colors = np.zeros_like(colors, dtype=np.uint8)
        palette_index = 0
        for color_index, color in enumerate(colors):
            if np.all(color == 0):
                recolored_colors[color_index] = [0, 0, 0]
                continue
            recolored_colors[color_index] = LABEL_PALETTE[palette_index % len(LABEL_PALETTE)]
            palette_index += 1
        return recolored_colors[inverse].reshape(mask_rgb.shape)

COMPONENT_COLORS = {
    'thing': '#1f77b4',
    'stuff_cluster': '#ff7f0e',
    'small_isolated': '#d62728',
}

if component_quality_df.empty:
    print('No component quality rows to plot')
else:
    plot_df = component_quality_df.copy()
    plot_df['row_label'] = plot_df['condition'].astype(str) + '/' + plot_df['image_id'].astype(str)
    ax = plot_df.set_index('row_label')[['num_thing', 'num_stuff_cluster', 'num_small_isolated']].plot(
        kind='bar',
        stacked=True,
        figsize=(14, 4),
        color=[COMPONENT_COLORS['thing'], COMPONENT_COLORS['stuff_cluster'], COMPONENT_COLORS['small_isolated']],
    )
    ax.set_title('component type composition from clean/generated grounding masks')
    ax.set_ylabel('node count')
    ax.tick_params(axis='x', rotation=60)
    plt.tight_layout()
    plt.show()

if component_score_df.empty:
    print('No component graph score rows to plot')
else:
    score_plot_df = component_score_df.groupby('condition')[['S_node', 'S_edge', 'S_unmatched', 'S_total', 'matched_ratio']].mean()
    ax = score_plot_df[['S_node', 'S_edge', 'S_unmatched', 'S_total']].plot(kind='bar', figsize=(11, 4))
    ax.set_title('clean reference vs generated shifted component graph scores')
    ax.set_ylabel('score')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

    ax = score_plot_df[['matched_ratio']].plot(kind='bar', figsize=(9, 3), color=['#2ca02c'])
    ax.set_ylim(0, 1.05)
    ax.set_title('component matching ratio by shifted condition')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()


def draw_component_nodes(ax, nodes: list[dict]) -> None:
    for node in nodes:
        x1, y1, x2, y2 = [float(value) for value in node['bbox']]
        node_type = str(node.get('node_type', '-'))
        color = COMPONENT_COLORS.get(node_type, '#666666')
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                max(1.0, x2 - x1),
                max(1.0, y2 - y1),
                fill=False,
                linewidth=1.5,
                edgecolor=color,
            )
        )
        cx, cy = [float(value) for value in node['centroid']]
        ax.scatter([cx], [cy], c=color, s=18)
        ax.text(x1, y1, f"{node_type} n={node.get('num_members', 1)}", color=color, fontsize=7)

VIS_COMPONENT_ROWS = min(6, len(mask_df))
for _, row in mask_df.head(VIS_COMPONENT_ROWS).iterrows():
    clean_row = clean_rows_by_id[str(row['base_image_id'])]
    generated_summary_path = query_summary_paths[str(row['condition'])]
    generated_summary = json.loads(generated_summary_path.read_text())
    generated_row = next(
        item for item in generated_summary['rows'] if str(item['base_image_id']) == str(row['base_image_id'])
    )

    clean_mask = recolor_label_mask(load_rgb(Path(clean_row['mask_path'])))
    generated_mask = recolor_label_mask(load_rgb(Path(generated_row['mask_path'])))
    shifted_image = load_rgb(Path(generated_row['source_path']))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    panels = [
        (clean_mask, clean_row['component_nodes'], f"clean components {row['base_image_id']}"),
        (generated_mask, generated_row['component_nodes'], f"generated mask components {row['condition']}"),
        (shifted_image, generated_row['component_nodes'], 'generated nodes on shifted image'),
    ]
    for ax, (image, nodes, title) in zip(axes, panels):
        ax.imshow(image)
        draw_component_nodes(ax, nodes)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## Cell Role: Visualize Clean Vs Shifted Generated Masks

각 row에 대해 original image, clean/reference mask, shifted image, expected mask, generated shifted mask, overlap map을 나란히 본다. overlap map은 green=true positive, red=missed expected, blue=extra generated foreground다.


In [ ]:
import matplotlib.pyplot as plt

VIS_MAX_ROWS = min(8, len(metrics_df))
LABEL_PALETTE = np.asarray([
    [31, 119, 180],
    [255, 127, 14],
    [44, 160, 44],
    [214, 39, 40],
    [148, 103, 189],
    [140, 86, 75],
    [227, 119, 194],
    [127, 127, 127],
    [188, 189, 34],
    [23, 190, 207],
], dtype=np.uint8)


def recolor_label_mask(mask_rgb: np.ndarray) -> np.ndarray:
    flat = mask_rgb.reshape(-1, 3)
    colors, inverse = np.unique(flat, axis=0, return_inverse=True)
    recolored_colors = np.zeros_like(colors, dtype=np.uint8)
    palette_index = 0
    for color_index, color in enumerate(colors):
        if np.all(color == 0):
            recolored_colors[color_index] = [0, 0, 0]
            continue
        recolored_colors[color_index] = LABEL_PALETTE[palette_index % len(LABEL_PALETTE)]
        palette_index += 1
    return recolored_colors[inverse].reshape(mask_rgb.shape)


def overlap_rgb(expected_fg: np.ndarray, generated_fg: np.ndarray) -> np.ndarray:
    output = np.zeros((*expected_fg.shape, 3), dtype=np.uint8)
    output[np.logical_and(expected_fg, generated_fg)] = [40, 200, 80]
    output[np.logical_and(expected_fg, ~generated_fg)] = [230, 60, 60]
    output[np.logical_and(~expected_fg, generated_fg)] = [60, 120, 240]
    return output


def expected_mask_for_row(row: pd.Series) -> Image.Image:
    clean_mask_image = Image.open(Path(row['clean_mask_path'])).convert('RGB')
    probe_row = mask_df[
        (mask_df['base_image_id'] == row['base_image_id'])
        & (mask_df['condition'] == row['condition'])
    ].iloc[0]
    if row['augmentation_type'] == 'position_shift':
        return position_shift_expected_mask(clean_mask_image, probe_row['params'], int(probe_row['seed']))
    return clean_mask_image

for _, row in metrics_df.head(VIS_MAX_ROWS).iterrows():
    probe_row = mask_df[
        (mask_df['base_image_id'] == row['base_image_id'])
        & (mask_df['condition'] == row['condition'])
    ].iloc[0]
    original = load_rgb(Path(probe_row['source_path']))
    shifted = load_rgb(Path(probe_row['shifted_path']))
    clean_mask = load_rgb(Path(row['clean_mask_path']))
    expected_mask = np.asarray(expected_mask_for_row(row).convert('RGB'))
    generated_mask = load_rgb(Path(row['generated_mask_path']))
    expected_fg = foreground_mask(expected_mask)
    generated_fg = foreground_mask(generated_mask)

    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    panels = [
        (original, f"original {row['base_image_id']}"),
        (recolor_label_mask(clean_mask), 'clean mask'),
        (shifted, row['condition']),
        (recolor_label_mask(expected_mask), 'expected mask'),
        (recolor_label_mask(generated_mask), 'generated DINO/SAM mask'),
        (overlap_rgb(expected_fg, generated_fg), f"overlap IoU={row['foreground_iou']:.3f}"),
    ]
    for ax, (image, title) in zip(axes, panels):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## Cell Role: Condition-Level Summary Plot

소수 샘플 기준으로 condition별 foreground IoU와 generated coverage가 어떻게 흔들리는지 빠르게 본다. 여기서 낮은 IoU나 coverage collapse가 보이면 해당 shift는 graph component input 자체가 불안정하다는 신호다.


In [ ]:
if metrics_df.empty:
    print('No metrics to plot')
else:
    summary_plot_df = metrics_df.groupby('condition')[['foreground_iou', 'coverage_ratio']].mean()
    ax = summary_plot_df.plot(kind='bar', figsize=(10, 4), color=['#1f77b4', '#ff7f0e'])
    ax.set_ylim(0, max(1.05, float(summary_plot_df.max().max()) * 1.1))
    ax.set_title('mean shifted GroundingDINO/SAM mask quality')
    ax.set_ylabel('score')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()
